In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset

In [ ]:
!pip install transformers datasets accelerate

In [ ]:
model = AutoModelForCausalLM.from_pretrained("roneneldan/TinyStories-1M")
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
MAX_LEN = 256

In [ ]:
dataset = load_dataset("IExploitableMan/smpl", split="train")
def tokenize_fn(example):
    prompt = example["question"] + tokenizer.eos_token
    answer = example["answer"] + tokenizer.eos_token
    full_text = prompt + answer
    tokens = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )
    input_ids = tokens["input_ids"]
    prompt_len = len(
        tokenizer(prompt, add_special_tokens=False)["input_ids"]
    )
    labels = input_ids.copy()
    labels[:prompt_len] = [-100] * prompt_len
    labels = [
        l if m == 1 else -100
        for l, m in zip(labels, tokens["attention_mask"])
    ]
    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": tokens["attention_mask"],
    }

dataset = dataset.map(tokenize_fn, batched=False)
dataset = dataset.remove_columns(["question", "answer"])

In [ ]:
training_args = TrainingArguments(
    output_dir="./out",
    per_device_train_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    num_train_epochs=40,
    logging_steps=10,
    save_steps=136,
    fp16=True,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=None,
)
trainer.train()

In [ ]:
input_text = "What is the largest organ in the body of the human?<|endoftext|>"

inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=10,
    do_sample=False,
)

print(tokenizer.decode(outputs[0]))